In [3]:
import numpy as np
import pandas as pd


class BooleanRetrieval:

    def __init__(self):
        self.index = {}
        self.documents_matrix = None

    def index_document(self, doc_id, text):
        terms = text.lower().split()
        print("Document -", doc_id, terms)

        for term in terms:
            if term not in self.index:
                self.index[term] = set()
            self.index[term].add(doc_id)

    def create_documents_matrix(self, documents):
        terms = list(self.index.keys())
        num_docs = len(documents)
        num_terms = len(terms)

        self.documents_matrix = np.zeros(
            (num_docs, num_terms), dtype=int
        )

        for i, (doc_id, text) in enumerate(documents.items()):
            doc_terms = text.lower().split()

            for term in doc_terms:
                if term in self.index:
                    term_id = terms.index(term)
                    self.documents_matrix[i, term_id] = 1

    def print_documents_matrix_table(self):
        df = pd.DataFrame(
            self.documents_matrix,
            columns=self.index.keys()
        )
        print("\nDocument-Term Matrix:")
        print(df)

    def print_all_terms(self):
        print("\nAll terms in the documents:")
        print(list(self.index.keys()))

    def boolean_search(self, query):
        tokens = query.lower().split()

        all_documents = set()
        for docs in self.index.values():
            all_documents.update(docs)

        result = None
        operator = "OR"

        for token in tokens:

            if token in ["and", "or", "not"]:
                operator = token.upper()

            else:
                term_docs = self.index.get(token, set())

                if result is None:
                    if operator == "NOT":
                        result = all_documents - term_docs
                    else:
                        result = term_docs.copy()

                elif operator == "AND":
                    result = result & term_docs

                elif operator == "OR":
                    result = result | term_docs

                elif operator == "NOT":
                    result = result - term_docs

                operator = "OR"

        return sorted(result) if result else []


if __name__ == "__main__":

    indexer = BooleanRetrieval()

    documents = {
        1: "Python is a programming language",
        2: "Information retrieval deals with finding information",
        3: "Boolean models are used in information retrieval"
    }

    for doc_id, text in documents.items():
        indexer.index_document(doc_id, text)

    indexer.create_documents_matrix(documents)
    indexer.print_documents_matrix_table()
    indexer.print_all_terms()

    query = input("\nEnter your boolean query: ")

    results = indexer.boolean_search(query)

    if results:
        print(f"Results for '{query}': {results}")
    else:
        print("No results found for the query.")

Document - 1 ['python', 'is', 'a', 'programming', 'language']
Document - 2 ['information', 'retrieval', 'deals', 'with', 'finding', 'information']
Document - 3 ['boolean', 'models', 'are', 'used', 'in', 'information', 'retrieval']

Document-Term Matrix:
   python  is  a  programming  language  information  retrieval  deals  with  \
0       1   1  1            1         1            0          0      0     0   
1       0   0  0            0         0            1          1      1     1   
2       0   0  0            0         0            1          1      0     0   

   finding  boolean  models  are  used  in  
0        0        0       0    0     0   0  
1        1        0       0    0     0   0  
2        0        1       1    1     1   1  

All terms in the documents:
['python', 'is', 'a', 'programming', 'language', 'information', 'retrieval', 'deals', 'with', 'finding', 'boolean', 'models', 'are', 'used', 'in']

Enter your boolean query: information OR deals
Results for 'informat